In [11]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

# If you plan on data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# If using Supabase
from supabase import create_client, Client

# 1.1 Connect to Supabase
load_dotenv(dotenv_path="../../.env")  # Adjust path if needed
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_ANON_KEY = os.getenv("SUPABASE_ANON_KEY")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)

pd.set_option("display.max_columns", None)  # optional, to see all columns

# 2.1. Query data from 2018–2023
response = (
    supabase
    .table("nba_historical_game_stats")
    .select("*")
    .gte("game_date", "2018-01-01")
    .lte("game_date", "2023-12-31")
    .execute()
)

data = response.data  # This is typically a list of dicts
df = pd.DataFrame(data)

# Print column names to see what's available
print("Available columns:", df.columns.tolist())

# Quick inspection
df.head()

# Get the total games played
total_games = len(df)
print(f"Total games in dataset: {total_games}")

# Calculate average scores
avg_home_score = df['home_score'].mean()
avg_away_score = df['away_score'].mean()
print(f"Average home score: {avg_home_score:.2f}")
print(f"Average away score: {avg_away_score:.2f}")

# Calculate home court advantage
home_court_advantage = avg_home_score - avg_away_score
print(f"Home court advantage: {home_court_advantage:.2f} points")

# Home win percentage
home_wins = (df['home_score'] > df['away_score']).sum()
home_win_pct = (home_wins / total_games) * 100
print(f"Home win percentage: {home_win_pct:.2f}%")

# Analyze scoring by quarter
quarters = ['q1', 'q2', 'q3', 'q4']
for q in quarters:
    home_avg = df[f'home_{q}'].mean()
    away_avg = df[f'away_{q}'].mean()
    diff = home_avg - away_avg
    print(f"Quarter {q[-1]}: Home {home_avg:.2f}, Away {away_avg:.2f}, Diff {diff:.2f}")

# Let's see which teams have the best home record
team_home_records = {}
for team in df['home_team'].unique():
    team_games = df[df['home_team'] == team]
    wins = (team_games['home_score'] > team_games['away_score']).sum()
    total = len(team_games)
    if total > 0:
        win_pct = (wins / total) * 100
        team_home_records[team] = {'wins': wins, 'games': total, 'win_pct': win_pct}

# Convert to DataFrame for better display
team_performance = pd.DataFrame.from_dict(team_home_records, orient='index')
team_performance.sort_values('win_pct', ascending=False).head(10)

# 4. Identify Target & Features
# 4.1. Define a "Home Team Win" Column
df["home_win"] = (df["home_score"] > df["away_score"]).astype(int)

# 4.2. Create useful features
# Team strength features (based on win percentages)
team_win_rates = {}

# Calculate home team win rates
for team in df['home_team'].unique():
    home_games = df[df['home_team'] == team]
    away_games = df[df['away_team'] == team]
    
    home_wins = home_games['home_win'].sum()
    away_wins = (away_games['home_win'] == 0).sum()
    
    total_games = len(home_games) + len(away_games)
    total_wins = home_wins + away_wins
    
    if total_games > 0:
        team_win_rates[team] = total_wins / total_games

# Add win rate features to each game
df['home_team_win_rate'] = df['home_team'].map(team_win_rates)
df['away_team_win_rate'] = df['away_team'].map(team_win_rates)

# Create scoring tendency features
df['home_offense_rating'] = df['home_score'] / df['home_score'].mean()
df['away_offense_rating'] = df['away_score'] / df['away_score'].mean()

# Feature for win streak (last 5 games)
# (This would require some more complex window functions)

# 5. Train/Validation Split
# 5.1 Time-Based Split
feature_cols = ['home_team_win_rate', 'away_team_win_rate', 
                'home_offense_rating', 'away_offense_rating']
target_col = 'home_win'

# Remove rows with missing values
df_model = df[feature_cols + [target_col] + ['game_date']].dropna()

# Split based on date
train_df = df_model[df_model["game_date"] < "2022-01-01"] 
test_df = df_model[df_model["game_date"] >= "2022-01-01"]
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# 5.2. X & y
X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

# 6. Model Training
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Predictions
train_preds = model.predict(X_train)
test_preds = model.predict(X_test)
test_probs = model.predict_proba(X_test)[:, 1]

# Evaluation
print("\nTraining Accuracy:", accuracy_score(y_train, train_preds))
print("Test Accuracy:", accuracy_score(y_test, test_preds))
print("Test AUC:", roc_auc_score(y_test, test_probs))
print("\nClassification Report:\n", classification_report(y_test, test_preds))

# Feature importance
coefficients = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_[0]
})
coefficients.sort_values('Coefficient', ascending=False)

Available columns: ['id', 'game_id', 'home_team', 'away_team', 'home_score', 'away_score', 'home_q1', 'home_q2', 'home_q3', 'home_q4', 'home_ot', 'away_q1', 'away_q2', 'away_q3', 'away_q4', 'away_ot', 'game_date', 'updated_at', 'home_assists', 'home_steals', 'home_blocks', 'home_turnovers', 'home_fouls', 'away_assists', 'away_steals', 'away_blocks', 'away_turnovers', 'away_fouls', 'home_off_reb', 'home_def_reb', 'home_total_reb', 'away_off_reb', 'away_def_reb', 'away_total_reb', 'home_3pm', 'home_3pa', 'away_3pm', 'away_3pa']
Total games in dataset: 1000
Average home score: 113.31
Average away score: 111.90
Home court advantage: 1.41 points
Home win percentage: 54.40%
Quarter 1: Home 28.88, Away 28.43, Diff 0.45
Quarter 2: Home 28.63, Away 28.41, Diff 0.22
Quarter 3: Home 28.68, Away 27.94, Diff 0.74
Quarter 4: Home 27.12, Away 27.12, Diff 0.00
Train shape: (291, 6)
Test shape: (708, 6)

Training Accuracy: 0.9347079037800687
Test Accuracy: 0.9661016949152542
Test AUC: 0.996619522860577

,Feature,Coefficient
2,home_offense_rating,5.191249
0,home_team_win_rate,0.945740
1,away_team_win_rate,-0.597850
3,away_offense_rating,-4.466295
